# 📚 Panini FIFA World Cup 2026 Album Manager

End-to-end tool to track your Panini sticker album using AI vision (OpenRouter).

## ⚠️ Requirements (read before running)

1. **OpenRouter API key** — get one at https://openrouter.ai/keys. You'll paste it when cell 2️⃣ asks for it (never stored in the notebook).
2. **OpenRouter credit** — top up your account before starting. Costs vary by model (see cell 2️⃣). Check current pricing at https://openrouter.ai/models.
3. **Google Drive access** — Colab will ask permission to mount your Drive when cell 1️⃣ runs. The album lives at `MyDrive/PaniniAlbum2026/`.
4. **Photos taken consistently:**
   - **Album pages** (countries + special): album lying flat, phone hovering above. Photos typically come **upside down** (180° rotation) — the notebook handles that automatically.
   - **Country pages**: one page per photo (each page = one country, exactly 20 stickers).
   - **Duplicates**: photos of the **BACK** of duplicate stickers. Lay them flat, no stacking or overlapping. **Maximum 28 stickers per photo** — the notebook rejects photos exceeding this cap.
5. **HEIC support** — iPhone `.HEIC` files work natively.

## 📂 Folder structure in Drive (auto-created on first run)

```
MyDrive/PaniniAlbum2026/
├── album_inventory.xlsx                ← master inventory (auto-created)
├── album_photos/                       ← upload country pages here
├── special_page_photos/                ← upload FWC + Coca-Cola pages here
├── duplicate_photos/                   ← upload back-of-duplicates here (≤28 per photo)
├── processed_album/                    ← processed (moved automatically after Excel write)
├── processed_special_pages/            ← processed
├── processed_duplicates/               ← processed
├── backups/                            ← timestamped Excel backups (auto-created before every write)
└── logs/                               ← raw AI responses per photo (audit trail)
```

## 🔄 Workflow

- **First run:** execute every cell in order. Cell 4️⃣ creates the master Excel with all 994 stickers preloaded.
- **Normal use:** upload new photos to the right folder, then run cells 6️⃣ (countries), 7️⃣ (special), 8️⃣ (duplicates). Photos are processed, Excel is updated, photos move to `processed_*/`. Run cell 9️⃣ for reports.
- **If a cell has no new photos to process, it just prints "📭 No new photos" and does nothing — safe to re-run.**

## 🛡️ Safety guarantees

- **Auto-backups:** every Excel write creates a timestamped copy in `backups/` first. The most recent 100 backups are kept.
- **Move-after-save:** photos move to `processed_*/` only after the Excel write succeeds. If the write fails, the photo stays for retry.
- **Strict country-page validation:** a country page is accepted only if the AI returned exactly 20 stickers covering all slots 1-20 for one country, AND all codes exist in the inventory.
- **Strict special-page validation:** only valid FWC0–FWC19 and CC1–CC14 codes are accepted; country codes leaking in cause rejection.
- **Duplicate hard cap:** photos with >28 detected codes are rejected entirely (likely hallucination).
- **Duplicate quality threshold:** if >25% of detected codes in a duplicate photo are unknown, the photo is rejected (likely bad reading).
- **Empty duplicate rejection:** photos with zero readable codes are rejected and stay for retry.
- **Robust boolean parsing:** the AI sometimes returns `"false"` as a string — `parse_bool()` handles this correctly so empty slots are never marked as owned.
- **Hash-based dedup:** the `PhotoRegistry` sheet tracks SHA-256 hashes, so re-uploading the same photo under a different name still won't be double-counted.
- **No accidental un-marking:** the `owned=True` flag is never set back to False by the AI — only manually with `mark_owned(code, False)`.
- **`self_check()` function:** audit your inventory at any time to verify structural integrity (994 stickers, all codes unique, 48 countries × 20, FWC complete, CC complete, no negative counts).

## 1️⃣ Setup — install packages and mount Drive

In [ ]:
!pip install -q openai openpyxl pandas pillow pillow-heif

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

BASE_DIR = '/content/drive/MyDrive/PaniniAlbum2026'

FOLDERS = {
    'album':              f'{BASE_DIR}/album_photos',
    'special':            f'{BASE_DIR}/special_page_photos',
    'duplicates':         f'{BASE_DIR}/duplicate_photos',
    'processed_album':    f'{BASE_DIR}/processed_album',
    'processed_special':  f'{BASE_DIR}/processed_special_pages',
    'processed_dups':     f'{BASE_DIR}/processed_duplicates',
}
for folder in FOLDERS.values():
    os.makedirs(folder, exist_ok=True)

EXCEL_PATH = f'{BASE_DIR}/album_inventory.xlsx'

print("📁 Folder structure ready in Drive:")
for name, path in FOLDERS.items():
    print(f"   {name}: {path}")
print(f"\n📊 Excel: {EXCEL_PATH}")

## 2️⃣ Configure OpenRouter

Generate your API key at https://openrouter.ai/keys

The key is requested interactively (never saved to the notebook). For pricing, check https://openrouter.ai/models — costs vary by model and may change over time.

**Model suggestions (roughly cheapest → most accurate):**

| Model | Strengths | Weaknesses |
|-------|-----------|------------|
| `google/gemini-2.5-flash` | Very cheap, fast | Sometimes skips 1–2 slots per page |
| `anthropic/claude-haiku-4.5` | Good balance of speed/accuracy | — |
| `google/gemini-2.5-pro` | Strong, good with text | Slower than Flash |
| `anthropic/claude-sonnet-4.5` | Highest accuracy on names/accents | Most expensive of these |

Set the `MODEL` variable below to your choice.

In [ ]:
from getpass import getpass
from openai import OpenAI

OPENROUTER_API_KEY = getpass("🔑 OpenRouter API key: ")

# Recommended models (cheapest → most accurate):
#   "google/gemini-2.5-flash"        → cheap, sometimes skips slots
#   "anthropic/claude-haiku-4.5"     → good balance ✓ default
#   "google/gemini-2.5-pro"          → strong, good with text
#   "anthropic/claude-sonnet-4.5"    → highest accuracy on names/accents
MODEL = "anthropic/claude-haiku-4.5"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)
print(f"✅ Client ready with model: {MODEL}")

## 3️⃣ Official country list — FIFA World Cup 2026

48 national teams × 20 stickers each. Page numbers come from the album's "FIFA WORLD CUP 2026" index page.

**Slot convention per country:**
- Slot **1** = country crest (holographic sticker)
- Slot **2** = main goalkeeper
- Slot **13** = team photo
- Slots **3–12, 14–20** = individual players

**Other sections:**
- **FWC** (special): 20 stickers (FWC 00 to FWC 19) scattered across intro, host pages, and history.
- **Coca-Cola**: 14 stickers (CC1 to CC14) on the Coca-Cola page.

In [ ]:
# Mapping: country code → starting page in the album
COUNTRIES = {
    # Group A
    'MEX':  8, 'RSA': 10, 'KOR': 12, 'CZE': 14,
    # Group B
    'CAN': 16, 'BIH': 18, 'QAT': 20, 'SUI': 22,
    # Group C
    'BRA': 24, 'MAR': 26, 'HAI': 28, 'SCO': 30,
    # Group D
    'USA': 32, 'PAR': 34, 'AUS': 36, 'TUR': 38,
    # Group E
    'GER': 40, 'CUW': 42, 'CIV': 44, 'ECU': 46,
    # Group F
    'NED': 48, 'JPN': 50, 'SWE': 52, 'TUN': 54,
    # Group G
    'BEL': 58, 'EGY': 60, 'IRN': 62, 'NZL': 64,
    # Group H
    'ESP': 66, 'CPV': 68, 'KSA': 70, 'URU': 72,
    # Group I
    'FRA': 74, 'SEN': 76, 'IRQ': 78, 'NOR': 80,
    # Group J
    'ARG': 82, 'ALG': 84, 'AUT': 86, 'JOR': 88,
    # Group K
    'POR': 90, 'COD': 92, 'UZB': 94, 'COL': 96,
    # Group L
    'ENG': 98, 'CRO': 100, 'GHA': 102, 'PAN': 104,
}
print(f"Total countries: {len(COUNTRIES)}")
assert len(COUNTRIES) == 48, "⚠️ Must be exactly 48 countries"
print(f"Country stickers: {48*20} = 960")
print(f"+ FWC: 20  + Coca-Cola: 14  →  TOTAL: 994")

## 4️⃣ Create the initial inventory

Runs only the first time. If the Excel already exists, it does NOT overwrite (your progress is preserved).

The inventory ships with **hardcoded metadata** for FWC and Coca-Cola stickers (player names / descriptions and page numbers), so even if the AI fails to identify a special sticker, the inventory already has the right info.

**Excel column names** (all in English):
- `code` — sticker code (no spaces): `ARG1`, `FWC9`, `CC1`
- `code_display` — pretty code with spaces: `ARG 1`, `FWC 09`, `CC1`
- `type` — `Crest`, `Goalkeeper`, `Team`, `Player`, `FWC`, `CocaCola`
- `section` — `ARG`, `FWC`, `Coca-Cola`, etc.
- `number` — slot number within the section
- `page` — album page number
- `player` — player name or sticker description
- `owned` — True/False, whether you have it
- `duplicates` — how many extra copies you have
- `date_obtained`

In [ ]:
import pandas as pd
from datetime import datetime

# ════════════════════════════════════════════════════════════
# HARDCODED METADATA FOR SPECIAL STICKERS (FWC and Coca-Cola)
# ════════════════════════════════════════════════════════════
# Page numbers and descriptions come from the actual album.
# Edit here if the publisher releases a different edition.

FWC_METADATA = [
    # (number, page, description)
    ( 0,   0, "Roll of Honour"),
    ( 1,   1, "Official Emblem"),
    ( 2,   1, "Official Emblem"),
    ( 3,   1, "Official Mascots"),
    ( 4,   1, "Official Slogan"),
    ( 5,   2, "Official Ball"),
    ( 6,   2, "Host Country Emblem CAN"),
    ( 7,   3, "Host Country Emblem MEX"),
    ( 8,   3, "Host Country Emblem USA"),
    ( 9, 106, "Italy 1934"),
    (10, 106, "Brazil 1950"),
    (11, 107, "Switzerland 1954"),
    (12, 107, "Chile 1962"),
    (13, 107, "Germany 1974"),
    (14, 108, "Mexico 1986"),
    (15, 108, "USA 1994"),
    (16, 109, "Korea/Japan 2002"),
    (17, 109, "Germany 2006"),
    (18, 109, "Brazil 2014"),
    (19, 109, "Qatar 2022"),
]

# Coca-Cola section: 14 player stickers across 2 pages (112 and 113)
CC_METADATA = [
    # (number, page, player name)
    ( 1, 112, "Lamine Yamal"),
    ( 2, 112, "Joshua Kimmich"),
    ( 3, 112, "Harry Kane"),
    ( 4, 112, "Santiago Giménez"),
    ( 5, 112, "Joško Gvardiol"),
    ( 6, 112, "Federico Valverde"),
    ( 7, 113, "Jefferson Lerma"),
    ( 8, 113, "Enner Valencia"),
    ( 9, 113, "Gabriel Magalhães"),
    (10, 113, "Virgil van Dijk"),
    (11, 113, "Alphonso Davies"),
    (12, 113, "Emiliano Martínez"),
    (13, 113, "Raúl Jiménez"),
    (14, 113, "Lautaro Martínez"),
]


def create_initial_inventory(excel_path, countries, force=False):
    """Create the initial Excel with all stickers + hardcoded FWC/CC metadata."""
    if os.path.exists(excel_path) and not force:
        print(f"⚠️ File already exists: {excel_path}")
        print("   Use force=True to recreate (a backup will be made first).")
        return

    # Auto-backup before destructive force=True overwrite
    # (backup_excel is defined later in the notebook; use raw shutil here)
    if os.path.exists(excel_path) and force:
        import shutil
        from datetime import datetime as _dt
        backup_dir = f'{BASE_DIR}/backups'
        os.makedirs(backup_dir, exist_ok=True)
        ts = _dt.now().strftime('%Y%m%d_%H%M%S_%f')
        backup_path = f'{backup_dir}/album_inventory_{ts}_before_force.xlsx'
        try:
            shutil.copy2(excel_path, backup_path)
            print(f"💾 Pre-force backup saved: {backup_path}")
        except Exception as e:
            print(f"⚠️ Could not back up before force: {e}")
            print("   ABORTING to protect your data. Pass force=False if you have no progress to lose.")
            return

    stickers = []

    # FWC (FWC 00 to FWC 19) — with hardcoded descriptions & pages
    for number, page, description in FWC_METADATA:
        stickers.append({
            'code':          f"FWC{number}",
            'code_display':  f"FWC {number:02d}",
            'type':          'FWC',
            'section':       'FWC',
            'number':        number,
            'page':          page,
            'player':        description,
            'owned':         False,
            'duplicates':    0,
            'date_obtained': '',
        })

    # Coca-Cola (CC1 to CC14) — with hardcoded player names and pages
    for number, page, player in CC_METADATA:
        stickers.append({
            'code':          f"CC{number}",
            'code_display':  f"CC{number}",
            'type':          'CocaCola',
            'section':       'Coca-Cola',
            'number':        number,
            'page':          page,
            'player':        player,
            'owned':         False,
            'duplicates':    0,
            'date_obtained': '',
        })

    # Countries (1–20 per country)
    for country, start_page in countries.items():
        for n in range(1, 21):
            if n == 1:
                slot_type = 'Crest'
            elif n == 2:
                slot_type = 'Goalkeeper'
            elif n == 13:
                slot_type = 'Team'
            else:
                slot_type = 'Player'
            page = start_page if n <= 10 else start_page + 1
            stickers.append({
                'code':          f"{country}{n}",
                'code_display':  f"{country} {n}",
                'type':          slot_type,
                'section':       country,
                'number':        n,
                'page':          page,
                'player':        '',
                'owned':         False,
                'duplicates':    0,
                'date_obtained': '',
            })

    df = pd.DataFrame(stickers)
    df.to_excel(excel_path, sheet_name='Inventory', index=False)

    print(f"✅ Inventory created with {len(stickers)} stickers")
    print(f"   • 20 FWC (with hardcoded names + pages)")
    print(f"   • 14 Coca-Cola (with hardcoded player names)")
    print(f"   • {len(countries)} countries × 20 = {len(countries)*20}")
    print(f"\n📊 Saved to: {excel_path}")

create_initial_inventory(EXCEL_PATH, COUNTRIES, force=False)

## 5️⃣ Vision helpers (OpenRouter)

**Rotation handling:**
- Album pages typically come **upside down** (the phone is held above the album) → rotated 180° by default.
- Duplicate photos: no rotation by default (the back labels usually come out readable).
- Both modern vision models (Gemini, Claude) read rotated text fine, but rotating first improves accuracy.

**HEIC support:** yes — iPhone `.HEIC` photos are decoded directly.

**If one specific photo comes in a weird orientation**, use:
```python
rotate_file('/content/drive/MyDrive/PaniniAlbum2026/album_photos/IMG_0731.HEIC', degrees=90)
```

**Auditing:** every call to the vision model stores its raw response in `MyDrive/PaniniAlbum2026/logs/` as JSON, so you can later inspect what the AI saw.

In [ ]:
import base64
import hashlib
import io
import json
import re
import shutil
from pathlib import Path
from PIL import Image, ImageOps

# iPhone HEIC support
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    print("✅ HEIC support enabled")
except ImportError:
    print("⚠️ pillow-heif not installed: .HEIC photos will fail")

# ═══════════════════════════════════════════════════════════════
# AUTOMATIC ROTATION
# ═══════════════════════════════════════════════════════════════
ROTATION_ALBUM      = 180
ROTATION_DUPLICATES = 0

# Max image dimension sent to the API (px on the longest side).
# Reduces cost and latency without losing legibility for sticker reading.
# iPhone photos are typically ~4000px on the long side; 2200 is plenty.
MAX_IMAGE_SIDE = 2200

def prepare_image(image_path, rotation=0, max_side=None):
    """Open the image, apply EXIF orientation, rotate if requested, and downsize.

    max_side: maximum dimension on the longest side (default: MAX_IMAGE_SIDE).
              Set to None or 0 to skip resizing.
    """
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img)
    if rotation:
        img = img.rotate(-rotation if rotation != 180 else 180, expand=True)
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    # Downsize if larger than max_side (in-place; preserves aspect ratio)
    limit = MAX_IMAGE_SIDE if max_side is None else max_side
    if limit and max(img.size) > limit:
        img.thumbnail((limit, limit), Image.LANCZOS)
    return img

def encode_image(image_path, rotation=0):
    """Encode image to base64 JPEG, applying EXIF + optional rotation."""
    img = prepare_image(image_path, rotation=rotation)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=88)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def rotate_file(image_path, degrees=180):
    """Rotate a file on disk. Useful if a specific photo came out wrong.

    degrees: 90, 180, or 270.
    """
    img = Image.open(image_path)
    img = ImageOps.exif_transpose(img)
    img = img.rotate(-degrees if degrees != 180 else 180, expand=True)
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    p = Path(image_path)
    if p.suffix.lower() in ('.heic', '.heif'):
        new_path = p.with_suffix('.jpg')
        img.save(new_path, quality=92)
        os.remove(image_path)
        print(f"✅ Rotated {degrees}° and converted to JPG: {new_path}")
        return str(new_path)
    img.save(image_path, quality=92)
    print(f"✅ Rotated {degrees}°: {image_path}")
    return image_path

def query_vision(image_path, prompt, client, model, max_retries=2, rotation=0):
    """Query the vision model with image + prompt."""
    b64 = encode_image(image_path, rotation=rotation)
    ext = "jpeg"
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {"type": "image_url",
                         "image_url": {"url": f"data:image/{ext};base64,{b64}"}}
                    ]
                }],
                temperature=0,
            )
            return response.choices[0].message.content
        except Exception as e:
            print(f"      ⚠️ Error attempt {attempt+1}: {e}")
    return None

def extract_json(text):
    """Extract the first JSON object from the response."""
    if not text:
        return None
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    return None

def list_images(folder):
    """List image files in a folder."""
    exts = ('.jpg', '.jpeg', '.png', '.heic', '.webp')
    return sorted([f for f in os.listdir(folder)
                   if f.lower().endswith(exts)])

# ═══════════════════════════════════════════════════════════════
# CODE NORMALIZATION
# ═══════════════════════════════════════════════════════════════
def normalize_code(c):
    """Normalize sticker codes.

    Handles all of these correctly:
        'FWC 09'   -> 'FWC9'
        'CC 01'    -> 'CC1'
        'ARG 01'   -> 'ARG1'
        'ECU-07'   -> 'ECU7'
        'arg 10'   -> 'ARG10'
        'KOR1'     -> 'KOR1'
        ''         -> ''
    """
    if not c:
        return ''
    c = str(c).upper().strip().replace(' ', '').replace('-', '')

    # FWC and CC: strip leading zeros (FWC09 -> FWC9)
    m = re.match(r'^(FWC|CC)0*(\d+)$', c)
    if m:
        return f"{m.group(1)}{int(m.group(2))}"

    # Country codes (3 letters + 1-2 digits): also strip leading zeros (ARG01 -> ARG1)
    m = re.match(r'^([A-Z]{3})0*(\d{1,2})$', c)
    if m:
        return f"{m.group(1)}{int(m.group(2))}"

    return c


def parse_bool(v):
    """Robust boolean parser for AI outputs.

    Handles the trap of bool("false") == True by treating common string
    representations explicitly. Returns False for anything not clearly truthy.

    Examples:
        parse_bool(True)         -> True
        parse_bool("true")       -> True
        parse_bool("false")      -> False    (the critical case)
        parse_bool("True")       -> True
        parse_bool("FALSE")      -> False
        parse_bool(1)            -> True
        parse_bool(0)            -> False
        parse_bool(None)         -> False
        parse_bool("")           -> False
        parse_bool("yes")        -> True
        parse_bool("filled")     -> True
    """
    if isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return bool(v)
    if isinstance(v, str):
        return v.strip().lower() in {
            'true', '1', 'yes', 'y', 'sí', 'si', 'filled', 'owned', 'pegado'
        }
    return False


def preview_image(image_path, rotation=None, max_size=900):
    """Show how an image will look after rotation."""
    if rotation is None:
        rotation = ROTATION_ALBUM
    img = prepare_image(image_path, rotation=rotation)
    img.thumbnail((max_size, max_size))
    from IPython.display import display
    display(img)
    print(f"Original size: {Image.open(image_path).size}")
    print(f"Rotation applied: {rotation}°")


# ═══════════════════════════════════════════════════════════════
# SAFE FILE MOVE (avoids overwriting existing files at destination)
# ═══════════════════════════════════════════════════════════════
def safe_move(src, dest_folder):
    """Move a file to dest_folder, avoiding overwrites.

    If a file with the same name already exists at the destination,
    a microsecond-precision timestamp is appended to the new name.
    Returns the final destination path.
    """
    os.makedirs(dest_folder, exist_ok=True)
    base = os.path.basename(src)
    dest = os.path.join(dest_folder, base)

    if not os.path.exists(dest):
        shutil.move(src, dest)
        return dest

    # Collision — append microsecond timestamp to avoid clobbering
    stem, ext = os.path.splitext(base)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    dest = os.path.join(dest_folder, f"{stem}_{timestamp}{ext}")
    shutil.move(src, dest)
    return dest


# ═══════════════════════════════════════════════════════════════
# FILE HASHING (for PhotoRegistry deduplication)
# ═══════════════════════════════════════════════════════════════
def file_hash(path, algo='sha256', block_size=65536):
    """Compute SHA-256 hash of a file (streamed for memory efficiency)."""
    h = hashlib.new(algo)
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


# ═══════════════════════════════════════════════════════════════
# AUTOMATIC BACKUPS (microsecond-precision timestamps)
# ═══════════════════════════════════════════════════════════════
BACKUPS_DIR = f'{BASE_DIR}/backups'
os.makedirs(BACKUPS_DIR, exist_ok=True)

# Keep at most this many backups (oldest deleted first)
MAX_BACKUPS_KEPT = 100

def backup_excel(excel_path, label='write'):
    """Create a timestamped backup of the Excel before any write.

    Uses microsecond precision in the timestamp, so two backups in
    the same second never collide.

    Returns the backup path, or None if the source did not exist yet.
    """
    if not os.path.exists(excel_path):
        return None
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    backup_name = f'album_inventory_{timestamp}_{label}.xlsx'
    backup_path = f'{BACKUPS_DIR}/{backup_name}'
    try:
        shutil.copy2(excel_path, backup_path)
    except Exception as e:
        print(f"      ⚠️ Backup failed ({e}). Proceeding with write anyway.")
        return None

    # Cap the number of backups
    try:
        existing = sorted([f for f in os.listdir(BACKUPS_DIR)
                           if f.endswith('.xlsx')])
        excess = len(existing) - MAX_BACKUPS_KEPT
        if excess > 0:
            for old in existing[:excess]:
                os.remove(f'{BACKUPS_DIR}/{old}')
    except Exception:
        pass  # housekeeping failure is non-fatal
    return backup_path


def save_inventory(df, excel_path, backup_label='write'):
    """Save only the Inventory sheet, preserving other sheets.
    Creates an Excel backup first.
    """
    backup_excel(excel_path, label=backup_label)
    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a',
                        if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name='Inventory', index=False)


# ═══════════════════════════════════════════════════════════════
# AUDITING AND LOGGING
# ═══════════════════════════════════════════════════════════════
LOGS_DIR = f'{BASE_DIR}/logs'
os.makedirs(LOGS_DIR, exist_ok=True)

def save_raw_response(photo_name, phase, response_text, parsed_data=None,
                      model=None):
    """Save raw AI response + parsed data as JSON, per photo."""
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
    base_name = Path(photo_name).stem
    log_path = f"{LOGS_DIR}/{phase}_{base_name}_{timestamp}.json"
    log_data = {
        'photo':         photo_name,
        'phase':         phase,
        'timestamp':     datetime.now().isoformat(),
        'model':         model,
        'raw_response':  response_text,
        'parsed_data':   parsed_data,
    }
    try:
        with open(log_path, 'w', encoding='utf-8') as f:
            json.dump(log_data, f, ensure_ascii=False, indent=2)
        return log_path
    except Exception as e:
        print(f"      ⚠️ Could not save JSON log: {e}")
        return None

def append_excel_log(excel_path, records):
    """Append rows to the ProcessingLog sheet of the Excel."""
    if not records:
        return
    df_new = pd.DataFrame(records)
    try:
        df_log = pd.read_excel(excel_path, sheet_name='ProcessingLog')
        df_log = pd.concat([df_log, df_new], ignore_index=True)
    except (ValueError, KeyError):
        df_log = df_new
    backup_excel(excel_path, label='log')
    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a',
                        if_sheet_exists='replace') as writer:
        df_log.to_excel(writer, sheet_name='ProcessingLog', index=False)

print("✅ Auditing helpers loaded (logs in " + LOGS_DIR + ")")
print(f"✅ Excel backups will be saved to {BACKUPS_DIR} (keeping latest {MAX_BACKUPS_KEPT})")
print(f"✅ Images will be resized to max {MAX_IMAGE_SIDE}px before sending to AI")

### 🔍 Preview rotation (optional but recommended)

Before processing all your photos, verify rotation looks right on ONE sample. If not, tweak `ROTATION_ALBUM` (0, 90, 180, or 270).

In [ ]:
# List available photos
available_photos = list_images(FOLDERS['album'])
print(f"Photos in album_photos/: {len(available_photos)}")
for f in available_photos[:5]:
    print(f"   • {f}")

# Preview the first photo with the configured rotation
if available_photos:
    first = os.path.join(FOLDERS['album'], available_photos[0])
    print(f"\n📷 Preview of: {available_photos[0]}")
    preview_image(first, rotation=ROTATION_ALBUM)
else:
    print("\n⚠️ Upload photos to the album_photos/ folder first")

## 6️⃣ Analyze country pages

📤 Upload your country page photos to `MyDrive/PaniniAlbum2026/album_photos/`.

**One photo = one country page = exactly 20 stickers.**

The AI extracts, per photo:
- Country code (3 letters)
- For each of the 20 slots: filled/empty + player name

### 🔒 Strict validation

A photo is **only accepted** if ALL of the following are true:
- The AI returned **exactly 20 stickers**
- All stickers share **the same 3-letter country code**
- Slots **1 through 20** are all covered, no duplicates
- All detected codes **exist in the master inventory** (no `unknown` codes)

If any check fails, the photo is **rejected** and stays in `album_photos/` for retry. The error is logged in `ProcessingLog` with details. Unknown codes typically mean the AI misread the country code (e.g. reading `BRA` as `BHA`); rejecting the whole page is safer than partial updates.

### Photo tips
- Good lighting, no glare
- Full page visible
- One country per photo

In [ ]:
PROMPT_ALBUM = """Photo of the Panini FIFA World Cup 2026 album — ONE COUNTRY PAGE.

IMPORTANT: the image may be rotated (90°, 180°, 270°). Read the content in its natural orientation (readable text).

═══════════════════════════════════════════════════════════
This is ONE country page (NOT two). It has EXACTLY 20 slots numbered 1-20
with a 3-letter country code (ARG, ECU, BRA, etc.).
→ You MUST always return ALL 20 SLOTS for the single country shown. No exceptions.
→ Each slot is either FILLED (sticker pasted on top) or EMPTY (large code visible).
→ There is NEVER a "cannot see" case: if the slot exists, return it.
→ When in doubt, prefer marking filled=false (a false negative is better than omitting).
→ All 20 stickers belong to the SAME country code.

SLOT STRUCTURE:
- Slot 1  = country crest (holographic sticker)
- Slot 2  = main goalkeeper (different shirt color)
- Slot 13 = team photo
- Rest    = individual players
═══════════════════════════════════════════════════════════

WHAT NOT TO INCLUDE (NOT STICKERS):
- Reference tables with groups A/B/C/D and flags (album index)
- Codes inside results/statistics grids (e.g. "1-0", "2-1")
- Medals, trophies, decorative balls without a code

HOW TO DETECT FILLED vs EMPTY:
- FILLED: you see the sticker image pasted on top (player photo, crest, team photo).
- EMPTY: you see the LARGE CODE (e.g. "ARG 7") on a flat colored background, with the player name printed near it.

HOW TO EXTRACT THE NAME:
- Filled player sticker: read the name printed on the sticker ("Lionel Messi", "Emiliano Martínez")
- Filled crest sticker: write "Crest"
- Filled team photo: write "Team"
- Empty player slot: name printed below the large code
- Preserve accents (á, é, ñ, ü) and "First Last" capitalization

VERIFICATION BEFORE RESPONDING:
- Did you include all 20 slots (1 to 20)? If not, add them all.
- Do all 20 stickers share the SAME 3-letter country code? They must.

Exact code WITHOUT spaces: "ARG13" not "ARG 13".

Respond with ONLY valid JSON, nothing before or after:
{
  "stickers": [
    {"code": "ARG1",  "filled": true,  "name": "Crest"},
    {"code": "ARG2",  "filled": true,  "name": "Emiliano Martínez"},
    {"code": "ARG13", "filled": true,  "name": "Team"},
    {"code": "ARG20", "filled": true,  "name": "Lionel Messi"}
  ]
}"""


def validate_country_page(stickers):
    """Validate that an AI response represents a complete country page.

    Returns (is_valid, error_message, country_code or None).

    Rules:
    - Exactly 20 stickers.
    - All stickers share the same 3-letter country code.
    - Slots 1-20 are all covered, no duplicates.
    """
    if len(stickers) != 20:
        return False, f"expected exactly 20 stickers, got {len(stickers)}", None

    countries_found = set()
    slots_found = []
    for s in stickers:
        code = normalize_code(s.get('code') or '')
        if not code:
            return False, "found sticker with empty code", None
        m = re.match(r'^([A-Z]{3})(\d{1,2})$', code)
        if not m:
            return False, f"code '{code}' does not look like a country sticker (XXX#)", None
        country = m.group(1)
        slot = int(m.group(2))
        countries_found.add(country)
        slots_found.append(slot)

    if len(countries_found) != 1:
        return False, f"expected 1 country, got {sorted(countries_found)}", None

    if sorted(slots_found) != list(range(1, 21)):
        missing = sorted(set(range(1, 21)) - set(slots_found))
        duplicates = sorted([s for s in slots_found if slots_found.count(s) > 1])
        msg_parts = []
        if missing:
            msg_parts.append(f"missing slots {missing}")
        if duplicates:
            msg_parts.append(f"duplicate slots {sorted(set(duplicates))}")
        return False, "; ".join(msg_parts), None

    return True, None, countries_found.pop()


def process_and_update_album(client, model, excel_path):
    """Process country page photos with STRICT validation + safe flow.

    For each photo:
      1. Send to the vision model.
      2. Save raw response as JSON (logs/).
      3. Validate strictly: must be exactly 20 stickers, slots 1-20, one country,
         AND all codes must exist in the master inventory.
      4. If invalid → REJECT (photo stays in album_photos/ for retry).
      5. If valid → update Excel (with auto-backup), then move photo with safe_move.
      6. Record the run in the ProcessingLog sheet.
    """
    photos = list_images(FOLDERS['album'])
    if not photos:
        print("📭 No new photos in album_photos/")
        return
    print(f"📸 Found {len(photos)} photos to analyze\n")

    today = datetime.now().strftime('%Y-%m-%d')
    full_timestamp = datetime.now().isoformat()
    log_records = []

    total_added = 0
    total_names = 0
    rejected_photos = []

    for photo in photos:
        path = os.path.join(FOLDERS['album'], photo)
        print(f"  🔍 {photo}")
        record = {
            'photo':        photo,
            'phase':        'album',
            'date':         full_timestamp,
            'model':        model,
            'success':      False,
            'n_stickers':   0,
            'n_filled':     0,
            'codes':        '',
            'errors':       '',
            'log_json':     '',
        }

        # 1) Send to vision model
        response = query_vision(path, PROMPT_ALBUM, client, model,
                                rotation=ROTATION_ALBUM)
        data = extract_json(response)

        # 2) Save raw response ALWAYS
        log_path = save_raw_response(photo, 'album', response, data, model=model)
        if log_path:
            record['log_json'] = log_path

        if not data or 'stickers' not in data:
            print(f"    ❌ Could not parse the response — photo stays in album_photos/")
            record['errors'] = 'unparseable JSON'
            log_records.append(record)
            rejected_photos.append(photo)
            continue

        stickers = data['stickers']
        record['n_stickers'] = len(stickers)
        record['n_filled']   = sum(1 for s in stickers if parse_bool(s.get('filled')))

        # 3a) Structural validation (count / country / slots)
        is_valid, err_msg, country = validate_country_page(stickers)
        if not is_valid:
            print(f"    🚫 REJECTED: {err_msg}")
            print(f"       Photo stays in album_photos/ for retry.")
            record['errors'] = f'REJECTED: {err_msg}'
            log_records.append(record)
            rejected_photos.append(photo)
            continue

        # 3b) Cross-check ALL detected codes exist in inventory
        try:
            df_check = pd.read_excel(excel_path, sheet_name='Inventory')
            known_codes = set(df_check['code'].tolist())
            detected = [normalize_code(s.get('code') or '') for s in stickers]
            unknown = [c for c in detected if c not in known_codes]
            if unknown:
                print(f"    🚫 REJECTED: {len(unknown)} detected codes not in inventory: {unknown}")
                print(f"       This usually means the AI misread the country code.")
                print(f"       Photo stays in album_photos/ for retry.")
                record['errors'] = f'REJECTED: unknown codes in inventory: {unknown}'
                log_records.append(record)
                rejected_photos.append(photo)
                continue
        except Exception as e:
            print(f"    ❌ Could not pre-check inventory: {e}")
            record['errors'] = f'Inventory check failed: {str(e)[:200]}'
            log_records.append(record)
            continue

        filled = record['n_filled']
        print(f"    ✅ {country}: 20/20 stickers ({filled} filled, {20-filled} empty)")

        # 4) Update Excel (atomic with backup)
        try:
            df = pd.read_excel(excel_path, sheet_name='Inventory')
            added = 0
            names_updated = 0
            detected_codes = []

            for s in stickers:
                code = normalize_code(s.get('code') or '')
                is_filled = parse_bool(s.get('filled'))
                name = (s.get('name') or '').strip()
                detected_codes.append(code)
                mask = df['code'] == code
                if not mask.any():
                    # Should never happen: we already verified above
                    continue
                idx = df.index[mask][0]
                if is_filled:
                    if not df.at[idx, 'owned']:
                        df.at[idx, 'date_obtained'] = today
                        added += 1
                    df.at[idx, 'owned'] = True
                if name:
                    current = df.at[idx, 'player']
                    if pd.isna(current) or str(current).strip() != name:
                        df.at[idx, 'player'] = name
                        names_updated += 1

            save_inventory(df, excel_path, backup_label='album')

            # 5) Move only after successful save (collision-safe)
            safe_move(path, FOLDERS['processed_album'])

            record['success'] = True
            record['codes']   = ','.join(detected_codes[:50])
            total_added += added
            total_names += names_updated

        except Exception as e:
            print(f"    ❌ Error updating Excel: {e}")
            record['errors'] = f'Excel: {str(e)[:200]}'
            # Photo stays in album_photos/ for retry

        log_records.append(record)

    # 6) Persist log
    append_excel_log(excel_path, log_records)

    print(f"\n✅ {total_added} new stickers marked as owned")
    print(f"📝 {total_names} names added/updated")
    print(f"📋 {len(log_records)} entries in ProcessingLog sheet")
    if rejected_photos:
        print(f"\n🚫 {len(rejected_photos)} photo(s) REJECTED (left in album_photos/):")
        for f in rejected_photos:
            print(f"     • {f}")
        print(f"\n   👉 Options for rejected photos:")
        print(f"      1. Inspect AI log: inspect_log('{rejected_photos[0]}', phase='album')")
        print(f"      2. Re-shoot the photo with better lighting / angle")
        print(f"      3. Try a more accurate model (e.g. claude-sonnet-4.5) and re-run this cell")

# Run
process_and_update_album(client, MODEL, EXCEL_PATH)

## 7️⃣ Analyze special pages (FWC + Coca-Cola)

📤 Upload photos of **non-country** pages to `MyDrive/PaniniAlbum2026/special_page_photos/`. These are pages with different layouts:

- **Intro pages** (FWC 00–04: Roll of Honour, Emblem, Mascots, Slogan…)
- **Host Countries and Cities** (FWC 5–8: Official Ball + Emblems CAN/MEX/USA)
- **FIFA World Cup History** (FWC 9–19: team photos of past World Cups)
- **Coca-Cola page** (CC1–CC14)

⚠️ The prompt is configured to **IGNORE**:
- The personalizable "MyPanini" sticker (PLAYER DATA)
- The mini "TOP SCORER" cards (these are printed design, not separate stickers)

### 🔒 Strict validation

A photo is **only accepted** if ALL of the following are true:
- At least 1 sticker was detected
- All codes are valid FWC (FWC0–FWC19) or CC (CC1–CC14) codes
- No country codes leaked in (e.g. ARG1)
- No duplicate codes within the photo

If any check fails, the photo is **rejected** and stays in `special_page_photos/`.

🌟 **Holographic hint:** all FWC stickers are **holographic** (metallic/iridescent finish with reflections). The prompt uses this as the primary signal to detect filled vs empty.

ℹ️ Even if the AI struggles to identify a particular special sticker, the inventory already has its name/description hardcoded from cell 4️⃣.

**Important:** these photos use the same rotation setting as regular album pages (`ROTATION_ALBUM = 180`).

In [ ]:
PROMPT_SPECIAL = """Photo of SPECIAL pages from the Panini FIFA World Cup 2026 album (NOT standard country pages).

IMPORTANT: the image may be rotated. Read the content in its natural orientation regardless of how it comes.

═══════════════════════════════════════════════════════════
TYPES OF STICKERS YOU MAY SEE:

A) FWC section (FIFA World Cup) — 20 special stickers numbered FWC 00 to FWC 19, scattered across several pages:
  • FWC 00       = Roll of Honour (gold sticker with list of World Cup champions)
  • FWC 1 to 4   = Emblems, mascots, and slogan of World Cup 2026
  • FWC 5        = Official Ball ("TRIONDA")
  • FWC 6, 7, 8  = Host Country Emblems (CAN, MEX, USA respectively)
  • FWC 9 to 19  = Team photos from 11 past World Cups (between 1930 and 2022)

  🌟 KEY HINT: ALL FWC stickers are HOLOGRAPHIC (metallic/iridescent finish with
  shimmer, reflections, and a mirror-like texture that shifts with the light). If you
  see that shiny metallic texture in the slot → it is FILLED. If you see a flat solid
  color background → it is EMPTY. This is the MOST reliable cue for FWC, more reliable
  than trying to read the artwork.

B) Coca-Cola section — 14 stickers CC1 to CC14:
  Red page with star player names (Lamine Yamal, Harry Kane, etc.) printed next to each
  slot. CC stickers are NOT holographic: they are regular player photos.
═══════════════════════════════════════════════════════════

HOW TO DETECT FILLED vs EMPTY:
- FWC FILLED: you see metallic/holographic shimmer (possibly with reflections, color
  shifts, iridescent texture). The pasted sticker covers the slot code. You do NOT need
  to recognize the image — just seeing the shiny holographic texture is enough.
- FWC EMPTY: you see the LARGE CODE (e.g. "FWC 9") on a flat/matte/solid background,
  with the description near it. There is NO holographic shimmer.
- CC FILLED: you see the player photo pasted (normal matte texture).
- CC EMPTY: you see the LARGE CODE (e.g. "CC5") on the red background, with the player
  name printed nearby.

WHAT TO PUT IN "name":
- CC stickers: the player name (e.g. "Lamine Yamal", "Harry Kane", "Lautaro Martínez").
- FWC 9-19 (team photos): the visible description of the World Cup (e.g. "Italy 1934",
  "Brazil 1950", "Qatar 2022"). Read it from the image, do not assume.
- FWC 00-8: the slot description (e.g. "Roll of Honour", "Official Emblem",
  "Official Ball", "Host Country Emblem CAN").
- If the sticker is FILLED but you cannot read the description because of the holographic
  shimmer, you may leave the name empty — what matters is marking filled=true correctly.
- Preserve accents (á, é, í, ó, ú, ñ, ü).

WHAT TO IGNORE (do NOT include in the response):
- The personalizable "MyPanini" sticker (text "PLAYER DATA", "NAME SURNAME", "Give here
  your MyPanini Sticker!"). NOT part of the official album.
- The small rectangular mini-cards labeled "TOP SCORER" or "TOP SCORERS" with small faces
  of old players. They are JUST PRINTED DESIGN, not stickers.
- Any code that is NOT FWC or CC (do not include country codes here).

CODE in the JSON: no spaces, uppercase. Examples: "FWC0" (not "FWC 00"), "FWC9", "CC1", "CC14".

Respond with ONLY valid JSON:
{
  "stickers": [
    {"code": "FWC0",  "filled": true,  "name": "Roll of Honour"},
    {"code": "FWC1",  "filled": true,  "name": "Official Emblem"},
    {"code": "FWC5",  "filled": false, "name": "Official Ball"},
    {"code": "FWC9",  "filled": false, "name": "Italy 1934"},
    {"code": "CC1",   "filled": false, "name": "Lamine Yamal"},
    {"code": "CC10",  "filled": true,  "name": "Virgil van Dijk"}
  ]
}"""


# Valid special-section codes
VALID_SPECIAL_CODES = (
    {f'FWC{i}' for i in range(20)} |   # FWC0 through FWC19
    {f'CC{i}' for i in range(1, 15)}   # CC1 through CC14
)


def validate_special_stickers(stickers):
    """Validate that an AI response only contains valid FWC / CC codes.

    Returns (is_valid, error_message).

    Rules:
    - At least 1 sticker.
    - All codes belong to VALID_SPECIAL_CODES (no country codes leaking in).
    - No duplicate codes within the same photo.
    """
    if not stickers:
        return False, "no stickers detected"

    codes = []
    for s in stickers:
        code = normalize_code(s.get('code') or '')
        if not code:
            return False, "found sticker with empty code"
        if code not in VALID_SPECIAL_CODES:
            return False, f"invalid special code: {code} (must be FWC0-19 or CC1-14)"
        codes.append(code)

    if len(codes) != len(set(codes)):
        from collections import Counter
        dups = [c for c, n in Counter(codes).items() if n > 1]
        return False, f"duplicate codes in same photo: {sorted(dups)}"

    return True, None


def process_and_update_special(client, model, excel_path):
    """Process special page photos with strict validation + safe flow."""
    photos = list_images(FOLDERS['special'])
    if not photos:
        print("📭 No new photos in special_page_photos/")
        return
    print(f"📸 Found {len(photos)} special page photos\n")

    today = datetime.now().strftime('%Y-%m-%d')
    full_timestamp = datetime.now().isoformat()
    log_records = []

    total_added = 0
    total_names = 0
    rejected_photos = []

    for photo in photos:
        path = os.path.join(FOLDERS['special'], photo)
        print(f"  🔍 {photo}")
        record = {
            'photo':       photo,
            'phase':       'special',
            'date':        full_timestamp,
            'model':       model,
            'success':     False,
            'n_stickers':  0,
            'n_filled':    0,
            'codes':       '',
            'errors':      '',
            'log_json':    '',
        }

        response = query_vision(path, PROMPT_SPECIAL, client, model,
                                rotation=ROTATION_ALBUM)
        data = extract_json(response)
        log_path = save_raw_response(photo, 'special', response, data, model=model)
        if log_path:
            record['log_json'] = log_path

        if not data or 'stickers' not in data:
            print(f"    ❌ Could not parse the response — photo stays in special_page_photos/")
            record['errors'] = 'unparseable JSON'
            log_records.append(record)
            rejected_photos.append(photo)
            continue

        stickers = data['stickers']
        record['n_stickers'] = len(stickers)
        record['n_filled']   = sum(1 for s in stickers if parse_bool(s.get('filled')))

        # Validate special-page codes
        is_valid, err_msg = validate_special_stickers(stickers)
        if not is_valid:
            print(f"    🚫 REJECTED: {err_msg}")
            print(f"       Photo stays in special_page_photos/ for retry.")
            record['errors'] = f'REJECTED: {err_msg}'
            log_records.append(record)
            rejected_photos.append(photo)
            continue

        filled = record['n_filled']
        print(f"    ✅ {len(stickers)} stickers detected ({filled} filled, "
              f"{len(stickers)-filled} empty)")

        try:
            df = pd.read_excel(excel_path, sheet_name='Inventory')
            added = 0
            names_updated = 0
            detected_codes = []

            for s in stickers:
                code = normalize_code(s.get('code') or '')
                is_filled = parse_bool(s.get('filled'))
                name = (s.get('name') or '').strip()
                if not code:
                    continue
                detected_codes.append(code)
                mask = df['code'] == code
                if not mask.any():
                    # Should not happen because we validated above
                    continue
                idx = df.index[mask][0]
                if is_filled:
                    if not df.at[idx, 'owned']:
                        df.at[idx, 'date_obtained'] = today
                        added += 1
                    df.at[idx, 'owned'] = True
                if name:
                    current = df.at[idx, 'player']
                    if pd.isna(current) or str(current).strip() != name:
                        df.at[idx, 'player'] = name
                        names_updated += 1

            save_inventory(df, excel_path, backup_label='special')

            safe_move(path, FOLDERS['processed_special'])

            record['success'] = True
            record['codes']   = ','.join(detected_codes[:50])
            total_added += added
            total_names += names_updated

        except Exception as e:
            print(f"    ❌ Error updating Excel: {e}")
            record['errors'] = f'Excel: {str(e)[:200]}'

        log_records.append(record)

    append_excel_log(excel_path, log_records)

    print(f"\n✅ {total_added} new special stickers marked as owned")
    print(f"📝 {total_names} descriptions added/updated")
    print(f"📋 {len(log_records)} entries in ProcessingLog sheet")
    if rejected_photos:
        print(f"\n🚫 {len(rejected_photos)} photo(s) REJECTED (left in special_page_photos/):")
        for f in rejected_photos:
            print(f"     • {f}")
        print(f"\n   👉 Options for rejected photos:")
        print(f"      1. Inspect AI log: inspect_log('{rejected_photos[0]}', phase='special')")
        print(f"      2. Re-shoot the photo with better lighting")
        print(f"      3. Try a more accurate model and re-run")

# Run
process_and_update_special(client, MODEL, EXCEL_PATH)

## 8️⃣ Analyze duplicates

📤 Upload photos of the **BACKS** of your duplicate stickers to `MyDrive/PaniniAlbum2026/duplicate_photos/`.

### 🚦 Hard limit: max 28 stickers per photo

The prompt is configured to **expect at most 28 stickers per photo**. Fewer is fine.

**What happens if a photo has more than 28 detected codes:**
- The photo is **REJECTED entirely** and **left in `duplicate_photos/`** for manual review.
- The notebook does NOT truncate, because there is no way to know which codes are real and which are AI hallucinations.
- The AI's full response is still saved to `logs/` so you can inspect what happened.
- A row is added to `ProcessingLog` with `success=FALSE` and a `REJECTED:` note.

**Your options when a photo is rejected:**
1. Inspect what the AI saw: `inspect_log('IMG_XXXX.HEIC', phase='duplicates')`
2. Re-shoot the photo with fewer stickers (split into multiple photos, ≤28 each)
3. Try a more accurate model (e.g. `claude-sonnet-4.5`) and re-run the cell

### Photo tips
- Lay stickers flat without overlapping
- If you have 3 copies of the same sticker in one photo, the AI counts all 3
- Make sure the white code rectangle is clearly visible on each sticker

### 🔐 Hash-based deduplication

Each photo's **SHA-256 hash and file size** are stored in the `PhotoRegistry` sheet (alongside the filename). If you re-upload the same physical photo — even renamed — it gets detected by hash and skipped (no double counting). To force re-processing a photo, manually remove its row from `PhotoRegistry`.

In [ ]:
PROMPT_DUPLICATES = """Photos of the BACKS of duplicate stickers from the Panini FIFA World Cup 2026 album.

IMPORTANT: the photo may be rotated in any orientation. Read the codes in their natural readable orientation regardless of how the photo is oriented.

Each sticker has a CODE printed on its back, inside a rounded white rectangle in a corner. Possible formats:
- Country + number (3 letters + 1-2 digits): "ECU 7", "KOR 1", "ARG 10", "SWE 13", "QAT 12", "BIH 9"...
- FWC + number: "FWC 0", "FWC 1", "FWC 10", "FWC 19"...
- CC + number: "CC 1", "CC 5", "CC 14"...

CRITICAL CONSTRAINT — MAXIMUM 28 STICKERS PER PHOTO:
- The user lays out a maximum of 28 stickers per photo.
- If you count MORE than 28 codes, you are HALLUCINATING. Re-check the image and remove the spurious ones.
- If the actual sticker count is unclear (overlap, blur, glare), it is BETTER to UNDER-count than over-count.

INSTRUCTIONS:
- List ALL clearly visible codes, one per sticker in the photo.
- If THIS photo has 3 stickers with the same code, include it 3 times (each physical sticker counts).
- If you cannot clearly read a code, OMIT it (better to skip than guess).
- Hard cap: 28 codes total. If your initial reading exceeds 28, re-examine the image and drop the least-confident ones.

CODE in the JSON: NO spaces, UPPERCASE. Examples: "KOR1", "QAT12", "FWC10", "CC5".

Respond with ONLY valid JSON, nothing before or after:
{
  "codes": ["KOR1", "FWC10", "ESP1", "CC5", "ECU7", "ECU7", "QAT12"]
}"""


# Maximum stickers the user can physically lay out per photo.
MAX_STICKERS_PER_DUPLICATE_PHOTO = 28


def process_and_update_duplicates(client, model, excel_path):
    """Process duplicate photos with safe flow + hash-based deduplication.

    Note: unlike country/special pages, duplicate photos with unknown codes are
    NOT rejected entirely — they update Excel with the known codes and only log
    the unknown ones. Rationale: one bad reading among 28 stickers should not
    discard the other 27 good ones.
    """
    from collections import Counter
    photos = list_images(FOLDERS['duplicates'])
    if not photos:
        print("📭 No new photos in duplicate_photos/")
        return
    print(f"📸 Found {len(photos)} duplicate photos\n")

    today = datetime.now().strftime('%Y-%m-%d')
    full_timestamp = datetime.now().isoformat()
    log_records = []

    try:
        df_registry = pd.read_excel(excel_path, sheet_name='PhotoRegistry')
        if 'hash' not in df_registry.columns:
            df_registry['hash'] = ''
        if 'size' not in df_registry.columns:
            df_registry['size'] = 0
        seen_hashes = set(h for h in df_registry['hash'].tolist() if h)
        seen_files  = set(df_registry['file'].tolist())
    except (ValueError, KeyError):
        df_registry = pd.DataFrame(columns=['file', 'hash', 'size', 'date', 'codes'])
        seen_hashes = set()
        seen_files  = set()

    total_added = 0
    total_unknown = []
    rejected_photos = []

    for photo in photos:
        path = os.path.join(FOLDERS['duplicates'], photo)
        print(f"  🔍 {photo}")
        record = {
            'photo':       photo,
            'phase':       'duplicates',
            'date':        full_timestamp,
            'model':       model,
            'success':     False,
            'n_stickers':  0,
            'n_filled':    0,
            'codes':       '',
            'errors':      '',
            'log_json':    '',
        }

        try:
            photo_hash = file_hash(path)
            photo_size = os.path.getsize(path)
        except Exception as e:
            print(f"    ❌ Could not hash photo: {e}")
            record['errors'] = f'hash failed: {e}'
            log_records.append(record)
            continue

        if photo_hash in seen_hashes:
            print(f"    ⏭️  same hash already processed, skipping and moving to processed/")
            safe_move(path, FOLDERS['processed_dups'])
            record['errors'] = 'already registered (hash match)'
            log_records.append(record)
            continue
        if photo in seen_files and photo_hash not in seen_hashes:
            print(f"    ⚠️ filename was processed before but content differs, treating as new")

        response = query_vision(path, PROMPT_DUPLICATES, client, model,
                                rotation=ROTATION_DUPLICATES)
        data = extract_json(response)

        log_path = save_raw_response(photo, 'duplicates', response, data, model=model)
        if log_path:
            record['log_json'] = log_path

        if not data or 'codes' not in data:
            print(f"    ❌ Could not parse response — photo stays in duplicate_photos/")
            record['errors'] = 'unparseable JSON'
            log_records.append(record)
            continue

        codes = [normalize_code(c) for c in data['codes']]
        codes = [c for c in codes if c]

        # (3) Reject photos where no codes could be read
        if not codes:
            print(f"    🚫 REJECTED: no readable codes detected")
            print(f"       Photo stays in duplicate_photos/ for manual review.")
            record['errors'] = 'REJECTED: no readable codes detected'
            log_records.append(record)
            rejected_photos.append(photo)
            continue

        if len(codes) > MAX_STICKERS_PER_DUPLICATE_PHOTO:
            print(f"    🚫 REJECTED: {len(codes)} codes detected, exceeds "
                  f"hard cap of {MAX_STICKERS_PER_DUPLICATE_PHOTO}.")
            print(f"       Photo stays in duplicate_photos/ for manual review.")
            print(f"       Detected codes: {', '.join(codes)}")
            record['errors'] = (f'REJECTED: {len(codes)} codes > '
                                f'{MAX_STICKERS_PER_DUPLICATE_PHOTO} (likely hallucination)')
            record['n_stickers'] = len(codes)
            record['codes'] = ','.join(codes[:50])
            log_records.append(record)
            rejected_photos.append(photo)
            continue

        print(f"    ✅ {len(codes)} codes: {', '.join(codes)}")

        # (5) Pre-check: if too many codes are unknown, reject the photo entirely.
        # Rationale: 1-2 misreads among 28 are tolerable, but if >25% of the codes
        # do not exist in the inventory, the AI is probably misreading the entire photo.
        try:
            df_precheck = pd.read_excel(excel_path, sheet_name='Inventory')
            known_codes = set(df_precheck['code'].tolist())
        except Exception as e:
            print(f"    ❌ Could not pre-check inventory: {e}")
            record['errors'] = f'Inventory check failed: {str(e)[:200]}'
            log_records.append(record)
            continue

        pre_unknown = [c for c in codes if c not in known_codes]
        unknown_ratio = len(pre_unknown) / max(len(codes), 1)
        UNKNOWN_REJECT_THRESHOLD = 0.25
        if len(pre_unknown) == len(codes) or unknown_ratio > UNKNOWN_REJECT_THRESHOLD:
            print(f"    🚫 REJECTED: {len(pre_unknown)}/{len(codes)} codes "
                  f"({unknown_ratio*100:.0f}%) not in inventory.")
            print(f"       Unknown: {pre_unknown[:10]}"
                  f"{' ...' if len(pre_unknown)>10 else ''}")
            print(f"       Threshold is {int(UNKNOWN_REJECT_THRESHOLD*100)}% — "
                  f"likely a bad AI reading. Photo stays in duplicate_photos/.")
            record['errors'] = (f'REJECTED: {len(pre_unknown)}/{len(codes)} unknown '
                                f'codes (>{int(UNKNOWN_REJECT_THRESHOLD*100)}% threshold)')
            record['n_stickers'] = len(codes)
            record['codes'] = ','.join(codes[:50])
            log_records.append(record)
            rejected_photos.append(photo)
            continue

        try:
            df = pd.read_excel(excel_path, sheet_name='Inventory')
            counter = Counter(codes)
            unknown = []
            known_added = 0  # (4) count only codes actually applied to inventory

            for code, qty in counter.items():
                mask = df['code'] == code
                if mask.any():
                    idx = df.index[mask][0]
                    if not df.at[idx, 'owned']:
                        df.at[idx, 'owned'] = True
                        df.at[idx, 'date_obtained'] = today
                    df.at[idx, 'duplicates'] = df.at[idx, 'duplicates'] + qty
                    known_added += qty
                else:
                    unknown.append(code)

            new_reg = pd.DataFrame([{
                'file':  photo,
                'hash':  photo_hash,
                'size':  photo_size,
                'date':  today,
                'codes': ','.join(codes),
            }])
            df_registry = pd.concat([df_registry, new_reg], ignore_index=True)
            seen_hashes.add(photo_hash)
            seen_files.add(photo)

            backup_excel(excel_path, label='duplicates')
            with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a',
                                if_sheet_exists='replace') as writer:
                df.to_excel(writer, sheet_name='Inventory', index=False)
                df_registry.to_excel(writer, sheet_name='PhotoRegistry', index=False)

            safe_move(path, FOLDERS['processed_dups'])

            record['success']    = True
            record['n_stickers'] = len(codes)
            record['codes']      = ','.join(codes[:50])
            if unknown:
                record['errors'] = f'codes not in inventory: {unknown[:10]}'
            # (4) Only count what was actually applied
            total_added += known_added
            total_unknown.extend(unknown)

        except Exception as e:
            print(f"    ❌ Error updating Excel: {e}")
            record['errors'] = f'Excel: {str(e)[:200]}'

        log_records.append(record)

    append_excel_log(excel_path, log_records)

    print(f"\n✅ {total_added} stickers added to inventory")
    print(f"📋 {len(log_records)} entries in ProcessingLog sheet")
    if rejected_photos:
        print(f"\n🚫 {len(rejected_photos)} photo(s) REJECTED (left in duplicate_photos/):")
        for f in rejected_photos:
            print(f"     • {f}")
        print(f"\n   👉 Manual review options:")
        print(f"      1. Inspect the AI log: inspect_log('{rejected_photos[0]}', phase='duplicates')")
        print(f"      2. Split the photo into smaller groups (max 28 each) and re-upload")
        print(f"      3. Try a different model (e.g. claude-sonnet-4.5) and re-run this cell")
    if total_unknown:
        unique = sorted(set(total_unknown))
        print(f"\n⚠️ Codes not found in inventory (logged but not blocking): {unique[:10]}"
              f"{' ...' if len(unique)>10 else ''}")
        print(f"   These are individual misreads — the rest of each photo was still processed.")

# Run
process_and_update_duplicates(client, MODEL, EXCEL_PATH)

## 9️⃣ Reports and statistics

Generates extra sheets in the Excel: **Missing**, **Duplicates**, **Stats**, **ByCountry**.

In [ ]:
def generate_reports(excel_path):
    """Generate auxiliary sheets: Missing, Duplicates, Stats, ByCountry."""
    df = pd.read_excel(excel_path, sheet_name='Inventory')

    missing = df[~df['owned']].copy()
    missing = missing[['code_display', 'type', 'section', 'number', 'player', 'page']]
    missing = missing.sort_values(['page', 'number'])

    dups = df[df['duplicates'] > 0].copy()
    dups = dups[['code_display', 'type', 'section', 'number', 'player', 'duplicates']]
    dups = dups.sort_values(['section', 'number'])

    total = len(df)
    owned = int(df['owned'].sum())
    missing_count = total - owned
    total_dups = int(df['duplicates'].sum())
    pct = (owned/total)*100

    by_country = df.groupby('section').agg(
        total=('owned', 'count'),
        owned=('owned', 'sum'),
    ).reset_index()
    by_country['missing'] = by_country['total'] - by_country['owned']
    by_country['percent'] = (by_country['owned']/by_country['total']*100).round(1)
    by_country = by_country.sort_values('percent', ascending=False)

    stats = pd.DataFrame([
        {'metric': 'Total stickers',     'value': total},
        {'metric': 'Owned',              'value': owned},
        {'metric': 'Missing',            'value': missing_count},
        {'metric': '% complete',         'value': f"{pct:.1f}%"},
        {'metric': 'Total duplicates',   'value': total_dups},
        {'metric': 'Slots with player',  'value': int((df['player'].fillna('').astype(str).str.len() > 0).sum())},
    ])

    backup_excel(excel_path, label='reports')
    with pd.ExcelWriter(excel_path, engine='openpyxl', mode='a',
                        if_sheet_exists='replace') as writer:
        df.to_excel(writer, sheet_name='Inventory', index=False)
        missing.to_excel(writer, sheet_name='Missing', index=False)
        dups.to_excel(writer, sheet_name='Duplicates', index=False)
        stats.to_excel(writer, sheet_name='Stats', index=False)
        by_country.to_excel(writer, sheet_name='ByCountry', index=False)

    print("="*50)
    print("📊 ALBUM SUMMARY")
    print("="*50)
    print(f"  Total:      {total}")
    print(f"  Owned:      {owned}  ({pct:.1f}%)")
    print(f"  Missing:    {missing_count}")
    print(f"  Duplicates: {total_dups}")
    print("="*50)
    print("\n🏆 Top 5 most complete countries:")
    print(by_country.head().to_string(index=False))
    print("\n🔻 Top 5 least complete countries:")
    print(by_country.tail().to_string(index=False))

generate_reports(EXCEL_PATH)

## 🔟 Trade lists

Copy-paste-ready lists for WhatsApp/Telegram trading groups. Includes player names so it's easy to identify stickers visually.

In [ ]:
def trade_list(excel_path):
    """Print copy-paste-ready trade lists."""
    df = pd.read_excel(excel_path, sheet_name='Inventory')
    dups = df[df['duplicates'] > 0].sort_values(['section', 'number'])
    missing = df[~df['owned']].sort_values(['section', 'number'])

    def fmt(row, is_dup=False):
        name = row['player'] if isinstance(row['player'], str) and row['player'].strip() else ''
        extra = f" x{int(row['duplicates'])}" if is_dup and row['duplicates'] > 1 else ""
        name_part = f" - {name}" if name else ""
        return f"     • {row['code_display']}{name_part}{extra}"

    print("🔁 DUPLICATES (available to trade):")
    print("-" * 45)
    current = None
    for _, row in dups.iterrows():
        if row['section'] != current:
            current = row['section']
            print(f"\n  📁 {current}:")
        print(fmt(row, is_dup=True))

    print("\n\n❌ MISSING (need these):")
    print("-" * 45)
    current = None
    for _, row in missing.iterrows():
        if row['section'] != current:
            current = row['section']
            print(f"\n  📁 {current}:")
        print(fmt(row))

trade_list(EXCEL_PATH)

## 🔧 Manual corrections (optional)

If the AI got something wrong (misspelled name, slot misdetected, etc.), use these helpers:

In [ ]:
def mark_owned(code, owned=True):
    """Mark/unmark a sticker as owned. Accepts any reasonable code format."""
    df = pd.read_excel(EXCEL_PATH, sheet_name='Inventory')
    code = normalize_code(code)
    mask = df['code'] == code
    if not mask.any():
        print(f"❌ Code {code} not found"); return
    idx = df.index[mask][0]
    df.at[idx, 'owned'] = owned
    # Only set the date if marking as owned and the cell is empty/NaN
    if owned:
        current_date = df.at[idx, 'date_obtained']
        if pd.isna(current_date) or str(current_date).strip() == '':
            df.at[idx, 'date_obtained'] = datetime.now().strftime('%Y-%m-%d')
    save_inventory(df, EXCEL_PATH, backup_label='manual')
    print(f"✅ {code}: owned={owned}")

def set_duplicates(code, qty):
    """Set the number of duplicate copies for a sticker. Accepts any reasonable code format."""
    df = pd.read_excel(EXCEL_PATH, sheet_name='Inventory')
    code = normalize_code(code)
    mask = df['code'] == code
    if not mask.any():
        print(f"❌ Code {code} not found"); return
    idx = df.index[mask][0]
    df.at[idx, 'duplicates'] = qty
    save_inventory(df, EXCEL_PATH, backup_label='manual')
    print(f"✅ {code}: duplicates={qty}")

def set_player(code, name):
    """Fix the player name / description of a sticker. Accepts any reasonable code format."""
    df = pd.read_excel(EXCEL_PATH, sheet_name='Inventory')
    code = normalize_code(code)
    mask = df['code'] == code
    if not mask.any():
        print(f"❌ Code {code} not found"); return
    idx = df.index[mask][0]
    df.at[idx, 'player'] = name
    save_inventory(df, EXCEL_PATH, backup_label='manual')
    print(f"✅ {code}: player='{name}'")

def inspect_log(photo_name, phase='album'):
    """Show the raw AI response for a specific photo (for auditing).

    photo_name: filename, e.g. 'IMG_0733.HEIC'
    phase: 'album', 'special', or 'duplicates'
    """
    import glob
    base_name = Path(photo_name).stem
    logs = sorted(glob.glob(f"{LOGS_DIR}/{phase}_{base_name}_*.json"))
    if not logs:
        print(f"❌ No logs found for {phase}_{base_name}")
        return
    with open(logs[-1]) as f:
        log = json.load(f)
    print(f"📄 Log: {logs[-1]}")
    print(f"   Model: {log.get('model')}")
    print(f"   Timestamp: {log.get('timestamp')}")
    data = log.get('parsed_data') or {}
    if 'stickers' in data:
        print(f"\n   {len(data['stickers'])} stickers detected:")
        for s in data['stickers']:
            code_val = s.get('code', '??')
            filled_val = str(s.get('filled', '?'))
            name_val = s.get('name', '')
            print(f"     {code_val:8s} - filled: {filled_val:5s} - {name_val}")
        from collections import Counter
        codes = [s.get('code') for s in data['stickers']]
        dups = {c: n for c, n in Counter(codes).items() if n > 1}
        if dups:
            print(f"\n   ⚠️ Duplicate codes in response: {dups}")
    elif 'codes' in data:
        print(f"\n   {len(data['codes'])} codes detected:")
        from collections import Counter
        for code, n in Counter(data['codes']).items():
            print(f"     {code}: x{n}")

def list_backups(limit=10):
    """List the most recent Excel backups available for rollback."""
    if not os.path.exists(BACKUPS_DIR):
        print(f"❌ Backup folder does not exist: {BACKUPS_DIR}")
        return
    backups = sorted([f for f in os.listdir(BACKUPS_DIR) if f.endswith(".xlsx")],
                     reverse=True)
    if not backups:
        print(f"📭 No backups found in {BACKUPS_DIR}")
        return
    print(f"📦 {len(backups)} backups in {BACKUPS_DIR} (newest first):")
    for b in backups[:limit]:
        size_kb = os.path.getsize(f'{BACKUPS_DIR}/{b}') / 1024
        print(f"   • {b}  ({size_kb:.1f} KB)")
    if len(backups) > limit:
        print(f"   ... and {len(backups) - limit} more")

def restore_backup(backup_filename):
    """Restore a specific backup as the current inventory.

    backup_filename: just the filename (use list_backups() to see options).
    """
    backup_path = f'{BACKUPS_DIR}/{backup_filename}'
    if not os.path.exists(backup_path):
        print(f"❌ Backup not found: {backup_path}")
        return
    # Backup the current file before overwriting
    backup_excel(EXCEL_PATH, label='before_restore')
    shutil.copy2(backup_path, EXCEL_PATH)
    print(f"✅ Restored from: {backup_filename}")
    print(f"   The previous inventory was backed up as 'before_restore' just in case.")


def self_check(excel_path=None):
    """Audit the inventory Excel for structural integrity.

    Checks:
    - Total sticker count is 994 (48 countries × 20 + 20 FWC + 14 CC)
    - All codes are unique
    - All required columns are present
    - No negative duplicate counts
    - All countries have exactly 20 stickers each
    - FWC section has 20 stickers (FWC0–FWC19)
    - Coca-Cola section has 14 stickers (CC1–CC14)
    """
    if excel_path is None:
        excel_path = EXCEL_PATH
    try:
        df = pd.read_excel(excel_path, sheet_name='Inventory')
    except Exception as e:
        print(f"❌ Could not read inventory: {e}")
        return False

    errors = []
    warnings = []

    # Total count
    if len(df) != 994:
        errors.append(f"Expected 994 stickers, found {len(df)}")

    # Unique codes
    if not df['code'].is_unique:
        dupes = df[df['code'].duplicated()]['code'].tolist()
        errors.append(f"Duplicate codes found: {dupes[:10]}")

    # Required columns
    required_cols = {
        'code', 'code_display', 'type', 'section', 'number',
        'page', 'player', 'owned', 'duplicates', 'date_obtained'
    }
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        errors.append(f"Missing columns: {sorted(missing_cols)}")
        # Cannot continue safely without required columns
        print("❌ Self-check FAILED:")
        for e in errors:
            print(f"   • {e}")
        return False

    # Non-negative duplicates
    if (df['duplicates'] < 0).any():
        bad = df[df['duplicates'] < 0]['code'].tolist()
        errors.append(f"Negative duplicate counts on: {bad[:10]}")

    # Country breakdown
    country_sections = df[df['type'].isin(['Crest', 'Goalkeeper', 'Team', 'Player'])]
    countries = country_sections['section'].value_counts()
    if len(countries) != 48:
        warnings.append(f"Expected 48 countries, found {len(countries)}")
    wrong_size = countries[countries != 20]
    if len(wrong_size) > 0:
        errors.append(f"Countries with != 20 stickers: {wrong_size.to_dict()}")

    # FWC section
    fwc = df[df['section'] == 'FWC']
    if len(fwc) != 20:
        errors.append(f"FWC section has {len(fwc)} stickers, expected 20")
    expected_fwc = {f'FWC{i}' for i in range(20)}
    actual_fwc = set(fwc['code'].tolist())
    if actual_fwc != expected_fwc:
        missing = expected_fwc - actual_fwc
        extra = actual_fwc - expected_fwc
        if missing:
            errors.append(f"Missing FWC codes: {sorted(missing)}")
        if extra:
            errors.append(f"Unexpected FWC codes: {sorted(extra)}")

    # CC section
    cc = df[df['section'] == 'Coca-Cola']
    if len(cc) != 14:
        errors.append(f"Coca-Cola section has {len(cc)} stickers, expected 14")
    expected_cc = {f'CC{i}' for i in range(1, 15)}
    actual_cc = set(cc['code'].tolist())
    if actual_cc != expected_cc:
        missing = expected_cc - actual_cc
        extra = actual_cc - expected_cc
        if missing:
            errors.append(f"Missing CC codes: {sorted(missing)}")
        if extra:
            errors.append(f"Unexpected CC codes: {sorted(extra)}")

    # Report
    print("="*60)
    print("🔎 SELF-CHECK RESULTS")
    print("="*60)
    if errors:
        print("❌ ERRORS:")
        for e in errors:
            print(f"   • {e}")
    if warnings:
        print("⚠️ WARNINGS:")
        for w in warnings:
            print(f"   • {w}")
    if not errors and not warnings:
        print(f"✅ Inventory is consistent: {len(df)} stickers, all checks passed.")
        owned = int(df['owned'].sum())
        print(f"   Progress: {owned}/{len(df)} owned ({owned/len(df)*100:.1f}%)")
        print(f"   Duplicates: {int(df['duplicates'].sum())}")
    print("="*60)
    return len(errors) == 0


# Examples (uncomment to use):
# mark_owned('ARG 10', True)         # accepts spaces, leading zeros, dashes
# mark_owned('arg-01')                # → normalized to 'ARG1'
# set_duplicates('FWC 09', 3)         # → normalized to 'FWC9'
# set_player('ARG 20', 'Lionel Messi')
# inspect_log('IMG_0733.HEIC', phase='album')
# list_backups()
# restore_backup('album_inventory_20260511_193024_album.xlsx')
# self_check()                        # audit the inventory